<a href="https://colab.research.google.com/github/phemango/ai-tpm-knowledge-assistant-/blob/main/AI_TPM_Knowledge_Assistant_Prototype_CLEAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Powered Enterprise Knowledge Assistant

## AI TPM Portfolio Project — RAG Prototype

This notebook demonstrates a prototype enterprise knowledge assistant using Retrieval-Augmented Generation (RAG).

### Project goals

- Retrieve information from approved enterprise documents

- Enforce document authorization and freshness rules

- Generate answers grounded in retrieved evidence

- Provide deterministic source attribution

- Evaluate both positive and negative scenarios

- Demonstrate production-readiness thinking from an AI TPM perspective

### Prototype architecture

Documents → Ingestion → Chunking → Embeddings → Vector Search → Authorization & Freshness Filtering → LLM → Answer + Source

> **Note:** This is a fictional demonstration using non-confidential sample documents. Google Colab is used only as a development/prototyping environment, not as a production deployment.

## 1. Environment Setup

In [50]:
!pip install pypdf

In [51]:
import requests

pdf_url = "https://raw.githubusercontent.com/phemango/ai-tpm-knowledge-assistant-/main/data/vacation_policy_current.pdf"

response = requests.get(pdf_url)

print("Status code:", response.status_code)
print("PDF downloaded:", len(response.content), "bytes")

Status code: 200
PDF downloaded: 2288 bytes


## 2. Document Ingestion

Load the fictional enterprise documents from the GitHub repository and extract their text for downstream processing.

In [52]:
from pypdf import PdfReader
from io import BytesIO

pdf_file = BytesIO(response.content)
reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    text += page.extract_text() + "\n"

print(text)

ACME CORP — VACATION POLICY
Document ID: HR-VAC-001
Version: 3.0
Status: Current
Effective Date: January 1, 2026
Owner: Human Resources
Access: All Employees
VACATION ACCRUAL
Full-time employees receive 20 vacation days per calendar year.
VACATION CARRY-OVER
Employees may carry over up to 5 unused vacation days into the following calendar year.
Any vacation days above the 5-day carry-over limit expire at the end of the calendar year.
VACATION REQUESTS
Employees should submit vacation requests through the company HR system at least two weeks before the
requested time off whenever practical.
POLICY AUTHORITY
This document is the current authoritative source for Acme Corp vacation carry-over rules.




In [53]:
print("Pages:", len(reader.pages))
print("Characters extracted:", len(text))
print("Has text:", len(text.strip()) > 0)

Pages: 1
Characters extracted: 706
Has text: True


In [54]:
import requests
from pypdf import PdfReader
from io import BytesIO

pdf_files = [
    "vacation_policy_current.pdf",
    "vacation_policy_old.pdf",
    "parental_leave_policy.pdf",
    "security_incident_procedure.pdf",
    "executive_compensation.pdf",
    "office_cafeteria_menu.pdf"
]

documents = []

for filename in pdf_files:
    url = f"https://raw.githubusercontent.com/phemango/ai-tpm-knowledge-assistant-/main/data/{filename}"
    response = requests.get(url)

    if response.status_code != 200:
        print(f"FAILED: {filename}")
        continue

    reader = PdfReader(BytesIO(response.content))

    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"

    documents.append({
        "source": filename,
        "text": text.strip()
    })

print("Documents ingested:", len(documents))

for doc in documents:
    print(f"{doc['source']}: {len(doc['text'])} characters")

Documents ingested: 6
vacation_policy_current.pdf: 704 characters
vacation_policy_old.pdf: 749 characters
parental_leave_policy.pdf: 540 characters
security_incident_procedure.pdf: 731 characters
executive_compensation.pdf: 694 characters
office_cafeteria_menu.pdf: 518 characters


## 3. Chunking

Split the ingested documents into retrieval units.

For this prototype, each small demo document remains one chunk to preserve context. A production implementation would use a section-aware chunking strategy optimized for document structure and retrieval quality.

In [55]:
chunks = []

for doc in documents:
    text = doc["text"]
    source = doc["source"]

    # Keep the document together for now because our demo PDFs are small.
    # We will split larger documents later using a smarter section-based strategy.
    chunks.append({
        "source": source,
        "chunk_id": f"{source}_chunk_1",
        "text": text
    })

print("Total chunks:", len(chunks))

for chunk in chunks:
    print("\n---")
    print("Source:", chunk["source"])
    print("Chunk ID:", chunk["chunk_id"])
    print("Characters:", len(chunk["text"]))

Total chunks: 6

---
Source: vacation_policy_current.pdf
Chunk ID: vacation_policy_current.pdf_chunk_1
Characters: 704

---
Source: vacation_policy_old.pdf
Chunk ID: vacation_policy_old.pdf_chunk_1
Characters: 749

---
Source: parental_leave_policy.pdf
Chunk ID: parental_leave_policy.pdf_chunk_1
Characters: 540

---
Source: security_incident_procedure.pdf
Chunk ID: security_incident_procedure.pdf_chunk_1
Characters: 731

---
Source: executive_compensation.pdf
Chunk ID: executive_compensation.pdf_chunk_1
Characters: 694

---
Source: office_cafeteria_menu.pdf
Chunk ID: office_cafeteria_menu.pdf_chunk_1
Characters: 518


## 4. Metadata, Authority, and Access

Enrich each chunk with document metadata such as version, status, owner, and access level.

These metadata fields support enterprise controls such as:

- identifying the authoritative/current source

- excluding retired documents

- enforcing access restrictions

- tracing answers back to the source document

In [56]:
import re

metadata_chunks = []

for chunk in chunks:
    text = chunk["text"]

    def get_field(field_name):
        match = re.search(
            rf"{field_name}:\s*(.+)",
            text,
            re.IGNORECASE
        )
        return match.group(1).strip() if match else None

    status = get_field("Status")

    metadata_chunks.append({
        "source": chunk["source"],
        "chunk_id": chunk["chunk_id"],
        "text": chunk["text"],
        "document_id": get_field("Document ID"),
        "version": get_field("Version"),
        "status": status,
        "effective_date": get_field("Effective Date"),
        "owner": get_field("Owner"),
        "access": get_field("Access"),
        "is_authoritative": status.lower() == "current" if status else False
    })

print("Metadata-enriched chunks:", len(metadata_chunks))

for chunk in metadata_chunks:
    print("\n---")
    print("Source:", chunk["source"])
    print("Document ID:", chunk["document_id"])
    print("Version:", chunk["version"])
    print("Status:", chunk["status"])
    print("Owner:", chunk["owner"])
    print("Access:", chunk["access"])
    print("Authoritative:", chunk["is_authoritative"])

Metadata-enriched chunks: 6

---
Source: vacation_policy_current.pdf
Document ID: HR-VAC-001
Version: 3.0
Status: Current
Owner: Human Resources
Access: All Employees
Authoritative: True

---
Source: vacation_policy_old.pdf
Document ID: HR-VAC-001
Version: 2.0
Status: Retired
Owner: Human Resources
Access: All Employees
Authoritative: False

---
Source: parental_leave_policy.pdf
Document ID: HR-PL-002
Version: 1.2
Status: Current
Owner: Human Resources
Access: All Employees
Authoritative: True

---
Source: security_incident_procedure.pdf
Document ID: SEC-INC-004
Version: 2.1
Status: Current
Owner: Information Security
Access: Security Team and Authorized IT Personnel
Authoritative: True

---
Source: executive_compensation.pdf
Document ID: EXEC-COMP-009
Version: 1.0
Status: Current
Owner: Executive Human Resources
Access: Board Members and Specifically Authorized Executives
Authoritative: True

---
Source: office_cafeteria_menu.pdf
Document ID: FAC-CAF-003
Version: 1.0
Status: Current
O

In [57]:
for chunk in metadata_chunks:
    print(
        chunk["source"],
        "| Status:", chunk["status"],
        "| Access:", chunk["access"],
        "| Authoritative:", chunk["is_authoritative"]
    )

vacation_policy_current.pdf | Status: Current | Access: All Employees | Authoritative: True
vacation_policy_old.pdf | Status: Retired | Access: All Employees | Authoritative: False
parental_leave_policy.pdf | Status: Current | Access: All Employees | Authoritative: True
security_incident_procedure.pdf | Status: Current | Access: Security Team and Authorized IT Personnel | Authoritative: True
executive_compensation.pdf | Status: Current | Access: Board Members and Specifically Authorized Executives | Authoritative: True
office_cafeteria_menu.pdf | Status: Current | Access: All Employees | Authoritative: True


In [58]:
!pip install sentence-transformers

## 5. Embeddings

Convert each document chunk into a numerical vector using a pretrained embedding model.

The prototype uses `all-MiniLM-L6-v2` as a lightweight baseline suitable for free Colab experimentation.

These vectors allow the system to compare the semantic meaning of an employee's question with the content of available documents.

In [59]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [60]:
sample_text = metadata_chunks[0]["text"]

embedding = embedding_model.encode(sample_text)

print("Embedding created.")
print("Vector dimensions:", len(embedding))
print("First 10 values:", embedding[:10])

Embedding created.
Vector dimensions: 384
First 10 values: [-0.04230692 -0.01917406  0.02492647  0.03295477  0.00399628  0.0035348
  0.05518732 -0.04202589 -0.04749561  0.00716039]


In [61]:
for chunk in metadata_chunks:
    chunk["embedding"] = embedding_model.encode(chunk["text"])

print("Embeddings created:", len(metadata_chunks))

for chunk in metadata_chunks:
    print(
        chunk["source"],
        "→",
        len(chunk["embedding"]),
        "dimensions"
    )

Embeddings created: 6
vacation_policy_current.pdf → 384 dimensions
vacation_policy_old.pdf → 384 dimensions
parental_leave_policy.pdf → 384 dimensions
security_incident_procedure.pdf → 384 dimensions
executive_compensation.pdf → 384 dimensions
office_cafeteria_menu.pdf → 384 dimensions


In [62]:
!pip install faiss-cpu

## 6. Vector Database and Semantic Search

Store the document embeddings in a FAISS vector index so that employee questions can be matched with semantically similar document content.

For this prototype, FAISS provides a lightweight, free vector-search capability suitable for Colab experimentation.

> **Production consideration:** a production system would use an enterprise-managed vector database or search platform with appropriate scalability, security, availability, and operational controls.

In [63]:
import faiss

import numpy as np

embedding_matrix = np.array(

    [chunk["embedding"] for chunk in metadata_chunks]

).astype("float32")

dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embedding_matrix)

print("FAISS index created.")

print("Vector dimensions:", dimension)

print("Vectors in index:", index.ntotal)

FAISS index created.
Vector dimensions: 384
Vectors in index: 6


## 7. Retrieval, Authorization, and Freshness Filtering

Retrieve documents that are semantically relevant to the employee's question.

The prototype then applies deterministic business rules to:

1. Exclude retired or non-authoritative documents

2. Enforce the user's access permissions

3. Keep only sufficiently relevant evidence

Authorization and document authority are enforced before information is passed to the LLM.

In [64]:
query = "How many vacation days can I carry over?"

query_embedding = embedding_model.encode([query]).astype("float32")

distances, indices = index.search(query_embedding, k=3)

print("Query:", query)

print("\nTop 3 matches:")

for rank, (distance, idx) in enumerate(zip(distances[0], indices[0]), start=1):

    chunk = metadata_chunks[idx]

    print(f"\n{rank}. {chunk['source']}")

    print("Distance:", distance)

    print("Status:", chunk["status"])

Query: How many vacation days can I carry over?

Top 3 matches:

1. vacation_policy_current.pdf
Distance: 0.8250978
Status: Current

2. vacation_policy_old.pdf
Distance: 0.8384346
Status: Retired

3. parental_leave_policy.pdf
Distance: 1.5926647
Status: Current


In [65]:
print("Retrieved documents:\n")

for rank, idx in enumerate(indices[0], start=1):

    chunk = metadata_chunks[idx]

    print(f"\n{'=' * 60}")

    print(f"RANK {rank}: {chunk['source']}")

    print(f"Status: {chunk['status']}")

    print(f"Distance: {distances[0][rank-1]:.4f}")

    print("=" * 60)

    print(chunk["text"])

Retrieved documents:


RANK 1: vacation_policy_current.pdf
Status: Current
Distance: 0.8251
ACME CORP — VACATION POLICY
Document ID: HR-VAC-001
Version: 3.0
Status: Current
Effective Date: January 1, 2026
Owner: Human Resources
Access: All Employees
VACATION ACCRUAL
Full-time employees receive 20 vacation days per calendar year.
VACATION CARRY-OVER
Employees may carry over up to 5 unused vacation days into the following calendar year.
Any vacation days above the 5-day carry-over limit expire at the end of the calendar year.
VACATION REQUESTS
Employees should submit vacation requests through the company HR system at least two weeks before the
requested time off whenever practical.
POLICY AUTHORITY
This document is the current authoritative source for Acme Corp vacation carry-over rules.

RANK 2: vacation_policy_old.pdf
Status: Retired
Distance: 0.8384
ACME CORP — VACATION POLICY
Document ID: HR-VAC-001
Version: 2.0
Status: Retired
Effective Date: January 1, 2024
Retirement Date: Decembe

In [66]:
authorized_current_chunks = [
    chunk for chunk in metadata_chunks
    if chunk["is_authoritative"] and chunk["status"].lower() == "current"
]

print("Eligible documents:", len(authorized_current_chunks))

for chunk in authorized_current_chunks:
    print(
        chunk["source"],
        "| Status:", chunk["status"],
        "| Authoritative:", chunk["is_authoritative"]
    )

Eligible documents: 5
vacation_policy_current.pdf | Status: Current | Authoritative: True
parental_leave_policy.pdf | Status: Current | Authoritative: True
security_incident_procedure.pdf | Status: Current | Authoritative: True
executive_compensation.pdf | Status: Current | Authoritative: True
office_cafeteria_menu.pdf | Status: Current | Authoritative: True


In [67]:
user_role = "Employee"

def has_access(chunk, user_role):
    access = chunk["access"].lower()

    if access == "all employees":
        return True

    return False


accessible_chunks = [
    chunk for chunk in authorized_current_chunks
    if has_access(chunk, user_role)
]

print("Accessible documents:", len(accessible_chunks))

for chunk in accessible_chunks:
    print(
        chunk["source"],
        "| Access:", chunk["access"]
    )

Accessible documents: 3
vacation_policy_current.pdf | Access: All Employees
parental_leave_policy.pdf | Access: All Employees
office_cafeteria_menu.pdf | Access: All Employees


In [68]:
query = "How many vacation days can I carry over?"

query_embedding = embedding_model.encode([query]).astype("float32")

# Create a temporary FAISS index using only eligible documents
filtered_embeddings = np.array(
    [chunk["embedding"] for chunk in accessible_chunks]
).astype("float32")

filtered_index = faiss.IndexFlatL2(filtered_embeddings.shape[1])
filtered_index.add(filtered_embeddings)

# Search the filtered knowledge base
distances, indices = filtered_index.search(filtered_embeddings[:1] * 0 + query_embedding, k=3)

print("Query:", query)
print("\nTop matches after filtering:")

for rank, (distance, idx) in enumerate(zip(distances[0], indices[0]), start=1):
    chunk = accessible_chunks[idx]

    print(f"{rank}. {chunk['source']}")
    print("   Distance:", round(float(distance), 4))
    print("   Status:", chunk["status"])
    print("   Access:", chunk["access"])

Query: How many vacation days can I carry over?

Top matches after filtering:
1. vacation_policy_current.pdf
   Distance: 0.8251
   Status: Current
   Access: All Employees
2. parental_leave_policy.pdf
   Distance: 1.5927
   Status: Current
   Access: All Employees
3. office_cafeteria_menu.pdf
   Distance: 1.766
   Status: Current
   Access: All Employees


In [69]:
relevance_threshold = 1.0

relevant_chunks = []

for distance, idx in zip(distances[0], indices[0]):
    if distance <= relevance_threshold:
        relevant_chunks.append(accessible_chunks[idx])

print("Relevant documents:", len(relevant_chunks))

for chunk in relevant_chunks:
    print(chunk["source"])

Relevant documents: 1
vacation_policy_current.pdf


In [70]:
!pip install transformers sentencepiece

## 8. Grounded Answer Generation

Use a language model to generate a concise answer from the approved, retrieved evidence.

The LLM is not given unrestricted access to the document corpus. Only evidence that has passed retrieval, authorization, and authority checks is provided as context.

The prototype initially uses a lightweight open-source model so the experiment can remain free to run in Google Colab.

Source attribution is generated deterministically from document metadata rather than relying on the LLM to create citations.

In [71]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("LLM loaded successfully.")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM loaded successfully.


In [72]:
context = "\n\n".join(
    chunk["text"] for chunk in relevant_chunks
)

prompt = f"""
Answer the employee's question using only the information in the context below.

Context:
{context}

Question:
{query}

Answer:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)

outputs = llm_model.generate(
    **inputs,
    max_new_tokens=100
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Answer:", answer)

Answer: 5


In [73]:
prompt = f"""
You are an enterprise knowledge assistant.

Answer the employee's question using ONLY the approved context below.

Requirements:
- Give a complete, concise answer.
- Do not invent information.
- Include the relevant policy details.
- At the end, identify the source document and version.

Approved context:
{context}

Employee question:
{query}

Answer:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)

outputs = llm_model.generate(
    **inputs,
    max_new_tokens=150
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

5


In [74]:
print("CONTEXT SENT TO LLM:")
print(context)

print("\n" + "="*60)

print("PROMPT SENT TO LLM:")
print(prompt)

CONTEXT SENT TO LLM:
ACME CORP — VACATION POLICY
Document ID: HR-VAC-001
Version: 3.0
Status: Current
Effective Date: January 1, 2026
Owner: Human Resources
Access: All Employees
VACATION ACCRUAL
Full-time employees receive 20 vacation days per calendar year.
VACATION CARRY-OVER
Employees may carry over up to 5 unused vacation days into the following calendar year.
Any vacation days above the 5-day carry-over limit expire at the end of the calendar year.
VACATION REQUESTS
Employees should submit vacation requests through the company HR system at least two weeks before the
requested time off whenever practical.
POLICY AUTHORITY
This document is the current authoritative source for Acme Corp vacation carry-over rules.

PROMPT SENT TO LLM:

You are an enterprise knowledge assistant.

Answer the employee's question using ONLY the approved context below.

Requirements:
- Give a complete, concise answer.
- Do not invent information.
- Include the relevant policy details.
- At the end, identi

In [75]:
source_chunk = relevant_chunks[0]

source_info = (
    f"Source: {source_chunk['document_id']} "
    f"| Version: {source_chunk['version']} "
    f"| Status: {source_chunk['status']}"
)

print(source_info)

Source: HR-VAC-001 | Version: 3.0 | Status: Current


In [76]:
answer_prompt = f"""
Answer the employee's question using ONLY the approved context.

Give a complete answer in 1-2 sentences.
Do not give only a number.
Do not invent information.

Approved context:
{context}

Employee question:
{query}

Answer:
"""

inputs = tokenizer(
    answer_prompt,
    return_tensors="pt",
    truncation=True
)

outputs = llm_model.generate(
    **inputs,
    max_new_tokens=100
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Answer:", answer)
print(source_info)

Answer: 5
Source: HR-VAC-001 | Version: 3.0 | Status: Current


In [77]:
query = "Who is eligible for parental leave?"

query_embedding = embedding_model.encode([query]).astype("float32")

distances, indices = filtered_index.search(
    query_embedding,
    k=3
)

print("Query:", query)
print("\nTop matches after authorization filtering:")

for rank, (distance, idx) in enumerate(
    zip(distances[0], indices[0]),
    start=1
):
    chunk = accessible_chunks[idx]

    print(f"\n{rank}. {chunk['source']}")
    print("   Distance:", round(float(distance), 4))

Query: Who is eligible for parental leave?

Top matches after authorization filtering:

1. parental_leave_policy.pdf
   Distance: 0.7872

2. vacation_policy_current.pdf
   Distance: 1.469

3. office_cafeteria_menu.pdf
   Distance: 1.6071


In [78]:
best_chunk = accessible_chunks[indices[0][0]]

print("Retrieved document:")
print(best_chunk["text"])

print("\nMetadata:")
print("Document ID:", best_chunk["document_id"])
print("Version:", best_chunk["version"])
print("Status:", best_chunk["status"])
print("Owner:", best_chunk["owner"])
print("Access:", best_chunk["access"])

Retrieved document:
ACME CORP — PARENTAL LEAVE POLICY
Document ID: HR-PL-002
Version: 1.2
Status: Current
Effective Date: March 1, 2026
Owner: Human Resources
Access: All Employees
ELIGIBILITY
Eligible full-time employees may receive up to 12 weeks of paid parental leave following the birth, adoption, or
placement of a child.
REQUESTS
Employees should contact Human Resources and submit the required leave request through the HR system.
POLICY AUTHORITY
This document is the current authoritative source for Acme Corp parental leave eligibility and duration.

Metadata:
Document ID: HR-PL-002
Version: 1.2
Status: Current
Owner: Human Resources
Access: All Employees


In [79]:
context = best_chunk["text"]

prompt = f"""
Answer the employee's question using ONLY the approved context.

Give a complete, concise answer.
Do not invent information.

Approved context:
{context}

Employee question:
{query}

Answer:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)

outputs = llm_model.generate(
    **inputs,
    max_new_tokens=100
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Answer:", answer)

print(
    f"Source: {best_chunk['document_id']} "
    f"| Version: {best_chunk['version']} "
    f"| Status: {best_chunk['status']}"
)

Answer: Acme Corp.
Source: HR-PL-002 | Version: 1.2 | Status: Current


In [80]:
baseline_results = {
    "vacation_carryover": {
        "question": "How many vacation days can I carry over?",
        "retrieved_document": "vacation_policy_current.pdf",
        "retrieval_status": "PASS",
        "llm_answer": answer if query == "How many vacation days can I carry over?" else "5",
        "answer_quality": "FAIL - overly terse"
    },
    "parental_leave": {
        "question": "Who is eligible for parental leave?",
        "retrieved_document": "parental_leave_policy.pdf",
        "retrieval_status": "PASS",
        "llm_answer": "Acme Corp.",
        "answer_quality": "FAIL - incorrect answer"
    }
}

print("Baseline evaluation saved.")
print(baseline_results)

Baseline evaluation saved.
{'vacation_carryover': {'question': 'How many vacation days can I carry over?', 'retrieved_document': 'vacation_policy_current.pdf', 'retrieval_status': 'PASS', 'llm_answer': '5', 'answer_quality': 'FAIL - overly terse'}, 'parental_leave': {'question': 'Who is eligible for parental leave?', 'retrieved_document': 'parental_leave_policy.pdf', 'retrieval_status': 'PASS', 'llm_answer': 'Acme Corp.', 'answer_quality': 'FAIL - incorrect answer'}}


In [81]:
!pip install -q "transformers>=4.45.0" sentencepiece

## 9. Model Comparison and Evaluation

Compare a lightweight baseline model with a stronger open-source model using representative enterprise questions.

Evaluation focuses on:

- retrieval correctness

- answer correctness

- completeness

- grounding in approved evidence

- source traceability

The purpose of model comparison is not to select a model based on popularity or size alone. Production model selection should be based on measured performance against the target use cases, together with cost, latency, reliability, and security requirements.

In [82]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

new_model_name = "google/flan-t5-base"

new_tokenizer = AutoTokenizer.from_pretrained(new_model_name)
new_llm_model = AutoModelForSeq2SeqLM.from_pretrained(new_model_name)

print("New LLM loaded successfully:", new_model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


New LLM loaded successfully: google/flan-t5-base


In [83]:
query = "Who is eligible for parental leave?"

context = best_chunk["text"]

prompt = f"""
Answer the employee's question using ONLY the approved context.

Give a complete, concise answer in 1-2 sentences.
Do not invent information.

Approved context:
{context}

Employee question:
{query}

Answer:
"""

inputs = new_tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)

outputs = new_llm_model.generate(
    **inputs,
    max_new_tokens=100
)

new_answer = new_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Answer:", new_answer)

print(
    f"Source: {best_chunk['document_id']} "
    f"| Version: {best_chunk['version']} "
    f"| Status: {best_chunk['status']}"
)

Answer: full-time employees
Source: HR-PL-002 | Version: 1.2 | Status: Current


In [84]:
query = "How many vacation days can I carry over?"

# Retrieve the correct document from our authorized corpus
query_embedding = embedding_model.encode([query]).astype("float32")

distances, indices = filtered_index.search(
    query_embedding,
    k=3
)

best_chunk = accessible_chunks[indices[0][0]]
context = best_chunk["text"]

prompt = f"""
Answer the employee's question using ONLY the approved context.

Give a complete, concise answer in 1-2 sentences.
Do not invent information.

Approved context:
{context}

Employee question:
{query}

Answer:
"""

inputs = new_tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)

outputs = new_llm_model.generate(
    **inputs,
    max_new_tokens=100
)

new_answer = new_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Answer:", new_answer)

print(
    f"Source: {best_chunk['document_id']} "
    f"| Version: {best_chunk['version']} "
    f"| Status: {best_chunk['status']}"
)

Answer: 5
Source: HR-VAC-001 | Version: 3.0 | Status: Current


## 10. Negative and Security Testing

Test failure and adversarial scenarios to verify that the assistant fails safely.

Scenarios include:

- Unauthorized document access

- No authorized evidence

- Retired or outdated information

- Questions outside the knowledge base

Expected behavior is to refuse unsupported answers rather than expose restricted information or generate unsupported content.

These tests are critical for evaluating enterprise AI safety, trustworthiness, and production readiness.

In [85]:
query = "What is the security incident procedure?"

query_embedding = embedding_model.encode([query]).astype("float32")

distances, indices = filtered_index.search(
    query_embedding,
    k=3
)

print("Query:", query)
print("\nTop matches after authorization filtering:")

for rank, (distance, idx) in enumerate(
    zip(distances[0], indices[0]),
    start=1
):
    chunk = accessible_chunks[idx]

    print(f"\n{rank}. {chunk['source']}")
    print("   Distance:", round(float(distance), 4))
    print("   Access:", chunk["access"])
    print("   Status:", chunk["status"])

Query: What is the security incident procedure?

Top matches after authorization filtering:

1. vacation_policy_current.pdf
   Distance: 1.7681
   Access: All Employees
   Status: Current

2. parental_leave_policy.pdf
   Distance: 1.7822
   Access: All Employees
   Status: Current

3. office_cafeteria_menu.pdf
   Distance: 1.7994
   Access: All Employees
   Status: Current


In [86]:
relevance_threshold = 1.0

relevant_chunks = []

for distance, idx in zip(distances[0], indices[0]):

    if distance <= relevance_threshold:

        relevant_chunks.append(accessible_chunks[idx])

print("Relevant authorized documents:", len(relevant_chunks))

for chunk in relevant_chunks:

    print(chunk["source"])

Relevant authorized documents: 0


In [87]:
query = "How many vacation days can I carry over?"

query_embedding = embedding_model.encode([query]).astype("float32")

# Search ALL documents, including retired documents.
all_embeddings = np.array(
    [chunk["embedding"] for chunk in metadata_chunks]
).astype("float32")

all_index = faiss.IndexFlatL2(all_embeddings.shape[1])
all_index.add(all_embeddings)

distances, indices = all_index.search(
    query_embedding,
    k=3
)

print("Query:", query)
print("\nTop 3 matches BEFORE current/authoritative filtering:")

for rank, (distance, idx) in enumerate(
    zip(distances[0], indices[0]),
    start=1
):
    chunk = metadata_chunks[idx]

    print(f"\n{rank}. {chunk['source']}")
    print("   Distance:", round(float(distance), 4))
    print("   Status:", chunk["status"])
    print("   Version:", chunk["version"])

Query: How many vacation days can I carry over?

Top 3 matches BEFORE current/authoritative filtering:

1. vacation_policy_current.pdf
   Distance: 0.8251
   Status: Current
   Version: 3.0

2. vacation_policy_old.pdf
   Distance: 0.8384
   Status: Retired
   Version: 2.0

3. parental_leave_policy.pdf
   Distance: 1.5927
   Status: Current
   Version: 1.2


In [88]:
current_chunks = [
    chunk for chunk in metadata_chunks
    if chunk["is_authoritative"]
    and chunk["status"].lower() == "current"
]

print("Documents after current/authoritative filtering:", len(current_chunks))

for chunk in current_chunks:
    print(
        chunk["source"],
        "| Version:", chunk["version"],
        "| Status:", chunk["status"]
    )

Documents after current/authoritative filtering: 5
vacation_policy_current.pdf | Version: 3.0 | Status: Current
parental_leave_policy.pdf | Version: 1.2 | Status: Current
security_incident_procedure.pdf | Version: 2.1 | Status: Current
executive_compensation.pdf | Version: 1.0 | Status: Current
office_cafeteria_menu.pdf | Version: 1.0 | Status: Current


In [89]:
query = "What is the company's policy for working from Mars?"

query_embedding = embedding_model.encode([query]).astype("float32")

# Search only documents that are current, authoritative,
# and accessible to this employee.
test_embeddings = np.array(
    [chunk["embedding"] for chunk in accessible_chunks]
).astype("float32")

test_index = faiss.IndexFlatL2(test_embeddings.shape[1])
test_index.add(test_embeddings)

distances, indices = test_index.search(
    query_embedding,
    k=3
)

relevant_chunks = []

for distance, idx in zip(distances[0], indices[0]):
    if distance <= relevance_threshold:
        relevant_chunks.append(accessible_chunks[idx])

print("Query:", query)
print("Relevant authorized documents:", len(relevant_chunks))

if len(relevant_chunks) == 0:
    print("Expected behavior: NO ANSWER — insufficient authorized evidence.")
else:
    print("WARNING: Evidence was found:")
    for chunk in relevant_chunks:
        print(chunk["source"])

Query: What is the company's policy for working from Mars?
Relevant authorized documents: 0
Expected behavior: NO ANSWER — insufficient authorized evidence.


## 11. Prototype Summary

This prototype demonstrates an end-to-end Retrieval-Augmented Generation (RAG) workflow for an enterprise knowledge assistant.

### What was demonstrated

- Enterprise document ingestion

- Metadata extraction

- Document chunking

- Semantic embeddings

- Vector search with FAISS

- Authorization filtering

- Current/authoritative document filtering

- Grounded LLM answer generation

- Deterministic source attribution

- Model comparison

- Negative and security testing

- Safe no-answer behavior when sufficient evidence is unavailable

### Key AI TPM considerations

The prototype was designed with production concerns in mind, including security, data authority, freshness, evaluation, reliability, cost, and operational ownership.

This is a portfolio prototype using fictional, non-confidential data. Google Colab is used for development and experimentation; production deployment would require enterprise infrastructure, identity and access controls, monitoring, governance, and operational support.